# **Company database project**

This project aims to show the example of simple database structure and how to use it, so it can satisfy common questions that might a bussines ower have. 

***

## **Helping Functions and libraries**

In [23]:
import sqlite3
from sqlite3 import Error, Connection
import pandas as pd
import os


def create_connection(path: str, new_db: bool = False):

    if new_db:
        # only delete the database if it exists
        if os.path.isfile(path):
            print("removed database:", path)
            os.remove(path)

    connection = None
    try:
        connection = sqlite3.connect(path)
        print("Connection successful")
    except Error as e:
        print(f"The error '{e}' occurred")
    return connection



def execute_command(connection, cmd: str):
    """"Executes an sql command, without return value, e.g.,
        for creating a table or inserting values.
    """
    cursor = connection.cursor()
    try:
        cursor.execute(cmd)
        connection.commit()
        print("Command successful")
    except Error as e:
        print(f"The error '{e}' occurred")



def execute_read_query(connection: Connection, query: str):

    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except Error as e:
        print(f"The error '{e}' occurred")


# Creting connection to database
connection = create_connection("soc_net.sqlite", new_db=True)



removed database: soc_net.sqlite
Connection successful


***

## **1. Creting the structure of database**

Following tables are going to be placed in the database:

* CUSTOMER ( Card number, Name, Surname, Tel )
* DEPARTMENT ( Dep ID, Name Dep, Manager )
* PRODUCT ( Product number, Name, Price, Stock, Department ID )
* BILL ( Bill number, Card number, Product number, Quantity, Date )
* EMPLOYEE ( Soc Sec Number, Name, Surname, City, Tel, Department ID )

In [24]:
crete_table_customers = """
CREATE TABLE IF NOT EXISTS customers (
    card_num INTEGER PRIMARY KEY AUTOINCREMENT,
    name VARCHAR(50) NOT NULL,
    surname VARCHAR(50) NOT NULL,
    telephone VARCHAR(15) 
    );
"""

crete_table_department = """
CREATE TABLE IF NOT EXISTS department (
    dep_id INTEGER PRIMARY KEY AUTOINCREMENT,
    dep_name VARCHAR(50) NOT NULL,
    manager VARCHAR(50)
    );
"""

crete_table_product = """
CREATE TABLE IF NOT EXISTS product (
    product_num INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name VARCHAR(50) NOT NULL,
    price FLOAT,
    stock INTEGER,
    dep_id INTEGER NOT NULL,
    FOREIGN KEY (dep_id) REFERENCES department (dep_id)
    CHECK (price > 0 AND stock >= 0)
    );
"""

crete_table_bill = """
CREATE TABLE IF NOT EXISTS bill (
    bill_num INTEGER PRIMARY KEY AUTOINCREMENT,
    card_num INTEGER NOT NULL,
    product_num INTEGER NOT NULL,
    quantity INTEGER,
    date VARCHAR(50),
    FOREIGN KEY (card_num) REFERENCES customers (card_num),
    FOREIGN KEY (product_num) REFERENCES product (product_num)
    CHECK (quantity > 0)
    );
"""


crete_table_employee = """
CREATE TABLE IF NOT EXISTS employee (
    soc_sec_num INTEGER PRIMARY KEY AUTOINCREMENT,
    name VARCHAR(50) NOT NULL,
    surname VARCHAR(50) NOT NULL,
    telephone VARCHAR(15),
    city VARCHAR(50),
    dep_id INTEGER NOT NULL,
    FOREIGN KEY (dep_id) REFERENCES department (dep_id)
    );
"""

commands = [crete_table_customers, crete_table_department, crete_table_product, crete_table_bill, crete_table_employee]

for cmd in commands:
    execute_command(connection, cmd)
    table_name = cmd.split()[5]
    print (f" Structure of {table_name} :")
    display(pd.read_sql(f"SELECT *FROM {table_name} ", connection))

Command successful
 Structure of customers :


,card_num,name,surname,telephone


Command successful
 Structure of department :


,dep_id,dep_name,manager


Command successful
 Structure of product :


,product_num,product_name,price,stock,dep_id


Command successful
 Structure of bill :


,bill_num,card_num,product_num,quantity,date


Command successful
 Structure of employee :


,soc_sec_num,name,surname,telephone,city,dep_id


***

## **2.1 Inserting values into the tables**

With ready schema, we can insert values into the database, normally it would be done from files where the data was stored befor, but for the project its going to be manually inserted data by the INSERT INTO command.


In [25]:
insert_department = """
INSERT INTO department (dep_name, manager) VALUES 
('Electronics', 'John Smith'),
('Dairy', 'Alice Newman'),
('Bakery', 'Mark Baker'),
('Garden', 'Eve Flower'),
('Clothing', 'Peter Needle'),
('Toys', 'Kate Bear'),
('Cosmetics', 'Monica Mirror');
"""

insert_customares = """
INSERT INTO customers (name, surname, telephone) VALUES 
('Adam', 'Walker', '500100200'),
('Beata', 'Knight', '600200300'),
('Charles', 'Peace', '700300400'),
('Dorothy', 'Wellman', '800400500'),
('Edward', 'Scissorhands', '900500600'),
('Felicia', 'Sweet', '111222333'),
('Gregory', 'Miller', '444555666');
"""

insert_products = """
INSERT INTO product (product_name, price, stock, dep_id) VALUES 
('Smartphone', 799.00, 15, 1),
('Laptop', 1200.50, 8, 1),
('Bluetooth Headphones', 89.99, 25, 1),
('Organic Milk', 3.50, 100, 2),
('Cheddar Cheese', 5.20, 40, 2),
('Greek Yogurt', 1.50, 60, 2),
('Butter', 4.00, 30, 2),
('Rye Bread', 4.20, 50, 3),
('Croissant', 2.10, 30, 3),
('Baguette', 1.80, 40, 3),
('Apple Pie', 12.00, 5, 3),
('Chocolate Muffin', 3.50, 20, 3),
('Lawn Mower', 850.00, 5, 4),
('Garden Hose', 25.00, 15, 4),
('Shovel', 15.90, 10, 4),
('Flower Pot', 5.50, 50, 4),
('Seeds Pack', 2.00, 200, 4),
('Fertilizer', 18.00, 25, 4),
('Cotton T-shirt', 49.90, 200, 5),
('Blue Jeans', 89.00, 50, 5),
('Wool Sweater', 120.00, 15, 5),
('Socks 3-pack', 12.50, 100, 5),
('Winter Jacket', 250.00, 10, 5),
('Baseball Cap', 19.99, 30, 5),
('Leather Belt', 35.00, 20, 5),
('LEGO Set', 120.00, 30, 6),
('Teddy Bear', 25.00, 40, 6),
('Remote Control Car', 55.00, 12, 6),
('Puzzle 1000pcs', 18.50, 20, 6),
('Barbie Doll', 22.00, 35, 6),
('Board Game', 45.00, 15, 6),
('Jump Rope', 7.00, 50, 6),
('Toy Train', 65.00, 8, 6),
('Hair Shampoo', 15.99, 80, 7),
('Conditioner', 14.50, 60, 7),
('Face Cream', 25.00, 30, 7),
('Hand Soap', 4.50, 120, 7),
('Toothpaste', 3.99, 100, 7),
('Deodorant', 6.00, 45, 7),
('Lip Balm', 2.50, 70, 7),
('Shower Gel', 8.00, 55, 7),
('Sunscreen', 19.00, 25, 7);
"""

insert_bills = """
INSERT INTO bill (card_num, product_num, quantity, date) VALUES 
(1, 1, 1, '2026-03-10'), 
(1, 15, 2, '2026-03-10'),
(2, 4, 3, '2026-03-10'),
(2, 5, 1, '2026-03-10'),
(2, 40, 1, '2026-03-10'),
(3, 10, 5, '2026-03-11'),
(3, 11, 2, '2026-03-11'),
(4, 20, 1, '2026-03-11'),
(4, 25, 3, '2026-03-11'),
(4, 2, 1, '2026-03-11'),
(5, 30, 2, '2026-03-12'),
(5, 31, 1, '2026-03-12'),
(5, 8, 4, '2026-03-12'),
(5, 42, 1, '2026-03-12'),
(6, 35, 1, '2026-03-12'),
(6, 18, 10, '2026-03-12'),
(7, 40, 2, '2026-03-13'),
(7, 3, 1, '2026-03-13'),
(7, 22, 1, '2026-03-13'),
(7, 14, 1, '2026-03-13');
"""

insert_employees = """
INSERT INTO employee (name, surname, telephone, city, dep_id) VALUES 
('Robert', 'Lewis', '123456789', 'London', 1),
('Michael', 'Scott', '123000111', 'New York', 1),
('Jim', 'Halpert', '123999888', 'Scranton', 1),
('Margaret', 'Taylor', '987654321', 'Chicago', 2),
('Pam', 'Beesly', '987111222', 'Scranton', 2),
('Thomas', 'Cat', '555666777', 'Boston', 3),
('Dwight', 'Schrute', '555111000', 'Scranton', 3),
('Angela', 'Martin', '555222333', 'Scranton', 3),
('Kevin', 'Malone', '555444555', 'Scranton', 3),
('Olivia', 'Wood', '444333222', 'Seattle', 4),
('Stanley', 'Hudson', '444000999', 'Scranton', 4),
('Phyllis', 'Vance', '444888777', 'Scranton', 4),
('David', 'Low', '111000111', 'New York', 5),
('Kelly', 'Kapoor', '111222333', 'New York', 5),
('Ryan', 'Howard', '111444555', 'New York', 5),
('Oscar', 'Martinez', '111666777', 'New York', 5),
('Meredith', 'Palmer', '111888999', 'New York', 5),
('Sarah', 'Singer', '222888222', 'Los Angeles', 6),
('Creed', 'Bratton', '222111000', 'Los Angeles', 6),
('Kevin', 'Jump', '333777333', 'Los Angeles', 7),
('Toby', 'Flenderson', '333000111', 'Los Angeles', 7),
('Andy', 'Bernard', '333222444', 'Los Angeles', 7),
('Erin', 'Hannon', '333555666', 'Los Angeles', 7),
('Nellie', 'Bertram', '333888999', 'Los Angeles', 7),
('Gabe', 'Lewis', '333444222', 'Los Angeles', 7);
"""

commands = [insert_department, insert_customares, insert_products, insert_bills, insert_employees]

for cmd in commands:
    execute_command(connection, cmd)
    table_name = cmd.split()[2]
    print (f" Data in table {table_name} :")
    display(pd.read_sql(f"SELECT * FROM {table_name} LIMIT 5 ", connection))

Command successful
 Data in table department :


,dep_id,dep_name,manager
0,1,Electronics,John Smith
1,2,Dairy,Alice Newman
2,3,Bakery,Mark Baker
3,4,Garden,Eve Flower
4,5,Clothing,Peter Needle


Command successful
 Data in table customers :


,card_num,name,surname,telephone
0,1,Adam,Walker,500100200
1,2,Beata,Knight,600200300
2,3,Charles,Peace,700300400
3,4,Dorothy,Wellman,800400500
4,5,Edward,Scissorhands,900500600


Command successful
 Data in table product :


,product_num,product_name,price,stock,dep_id
0,1,Smartphone,799.00,15,1
1,2,Laptop,1200.50,8,1
2,3,Bluetooth Headphones,89.99,25,1
3,4,Organic Milk,3.50,100,2
4,5,Cheddar Cheese,5.20,40,2


Command successful
 Data in table bill :


,bill_num,card_num,product_num,quantity,date
0,1,1,1,1,2026-03-10
1,2,1,15,2,2026-03-10
2,3,2,4,3,2026-03-10
3,4,2,5,1,2026-03-10
4,5,2,40,1,2026-03-10


Command successful
 Data in table employee :


,soc_sec_num,name,surname,telephone,city,dep_id
0,1,Robert,Lewis,123456789,London,1
1,2,Michael,Scott,123000111,New York,1
2,3,Jim,Halpert,123999888,Scranton,1
3,4,Margaret,Taylor,987654321,Chicago,2
4,5,Pam,Beesly,987111222,Scranton,2


***

## **2.2 Modifying/adding attributes into the database's tables**

Now, with the basic structer and some data we can still modify the schema if we wnat to. To showcase this feature th following actions are going to be done:

* New decimal column DISCOUNT added into the customare table - value 1 symbolize the customare not having any discount (100% of the price), and value 0.95 symbolize the customare having discount of 5% (95% of price) for every shoping
* New character column SALARY_GROUP added into the employes table - there will be three levels of salary that a department employee can get:

    **A level** - 24.000$/year, **B level** - 28.000$/year and **C level** - 32.000$/year

* New character column ORIGINS added into product table - addressing whether the product was imported or is a local product. (Can be also Unknown)

In [26]:
new_column_discount = """
ALTER TABLE customers
ADD COLUMN discount DECIMAL DEFAULT 1
CHECK (discount IN (1, 0.95))
"""

new_column_salary = """
ALTER TABLE employee
ADD COLUMN salary_group CHAR(1) DEFAULT 'A'
CHECK (salary_group IN ('A', 'B', 'C'))
"""

new_column_origins  = """
ALTER TABLE product
ADD COLUMN origins VARCHAR(50) DEFAULT 'Unknown'
CHECK (origins IN ('Unknown', 'Local', 'Imported'))

"""
commands = [new_column_discount, new_column_salary, new_column_origins]

for cmd in commands:
    execute_command(connection, cmd)
    table_name = cmd.split()[2]
    print (f" Data in table {table_name} :")
    display(pd.read_sql(f"SELECT * FROM {table_name} LIMIT 5 ", connection))



Command successful
 Data in table customers :


,card_num,name,surname,telephone,discount
0,1,Adam,Walker,500100200,1
1,2,Beata,Knight,600200300,1
2,3,Charles,Peace,700300400,1
3,4,Dorothy,Wellman,800400500,1
4,5,Edward,Scissorhands,900500600,1


Command successful
 Data in table employee :


,soc_sec_num,name,surname,telephone,city,dep_id,salary_group
0,1,Robert,Lewis,123456789,London,1,A
1,2,Michael,Scott,123000111,New York,1,A
2,3,Jim,Halpert,123999888,Scranton,1,A
3,4,Margaret,Taylor,987654321,Chicago,2,A
4,5,Pam,Beesly,987111222,Scranton,2,A


Command successful
 Data in table product :


,product_num,product_name,price,stock,dep_id,origins
0,1,Smartphone,799.00,15,1,Unknown
1,2,Laptop,1200.50,8,1,Unknown
2,3,Bluetooth Headphones,89.99,25,1,Unknown
3,4,Organic Milk,3.50,100,2,Unknown
4,5,Cheddar Cheese,5.20,40,2,Unknown


Now with new columns, we hove only default values in them, so its better to 'simulate' some reakl data into them with update column, for this I am going to use modulo operator

In [27]:
new_discouts = """
UPDATE customers
SET discount =CASE 
    WHEN (ABS(RANDOM()) % 10) < 8 THEN 1      
    ELSE 0.95
END;
"""

new_salary_groups = """
UPDATE employee
SET salary_group = CASE
    WHEN (ABS(RANDOM()) % 10) < 6 THEN 'A'      
    WHEN (ABS(RANDOM()) % 10) < 8 THEN 'B'   
    ELSE 'C'
END;
"""

new_origins = """
UPDATE product
SET origins = CASE 
    WHEN (ABS(RANDOM()) % 10) < 6 THEN 'Local'      
    WHEN (ABS(RANDOM()) % 10) < 9 THEN 'Imported'   
    ELSE 'Unknown'                                  
END;
"""

commands = [new_discouts, new_salary_groups, new_origins]

for cmd in commands:
    execute_command(connection, cmd)
    table_name = cmd.split()[1]
    print (f" NEW Data in table {table_name} :")
    display(pd.read_sql(f"SELECT * FROM {table_name} LIMIT 5 ", connection))




Command successful
 NEW Data in table customers :


,card_num,name,surname,telephone,discount
0,1,Adam,Walker,500100200,0.95
1,2,Beata,Knight,600200300,0.95
2,3,Charles,Peace,700300400,1.00
3,4,Dorothy,Wellman,800400500,1.00
4,5,Edward,Scissorhands,900500600,1.00


Command successful
 NEW Data in table employee :


,soc_sec_num,name,surname,telephone,city,dep_id,salary_group
0,1,Robert,Lewis,123456789,London,1,B
1,2,Michael,Scott,123000111,New York,1,A
2,3,Jim,Halpert,123999888,Scranton,1,A
3,4,Margaret,Taylor,987654321,Chicago,2,A
4,5,Pam,Beesly,987111222,Scranton,2,C


Command successful
 NEW Data in table product :


,product_num,product_name,price,stock,dep_id,origins
0,1,Smartphone,799.00,15,1,Local
1,2,Laptop,1200.50,8,1,Local
2,3,Bluetooth Headphones,89.99,25,1,Local
3,4,Organic Milk,3.50,100,2,Local
4,5,Cheddar Cheese,5.20,40,2,Local


***
## **3.Modifying the manager and the employees of a department**

The people working in the company are changing, usefull feature is to delete/add managers and employyes in the table. Within this coe the following situation is simulated:

*The manager of Electronics department has resign. Two of his most loyal employees has decided to leave as well since he is no longer a manager. The HR department promoted one of the employees of electronics department to manager and hired three more people as replecment of the ones who leaved or got promoted*

In [28]:
changing_manager ="""
UPDATE department
SET manager = 'Michael Scott'
WHERE dep_name = 'Electronics';
"""

deleting_employees = """
DELETE FROM employee 
WHERE (name, surname, dep_id) IN (
    ('Michael', 'Scott', (SELECT dep_id FROM department WHERE dep_name = 'Electronics')),
    ('Robert', 'Lewis', (SELECT dep_id FROM department WHERE dep_name = 'Electronics')),
    ('Jim', 'Halpert', (SELECT dep_id FROM department WHERE dep_name = 'Electronics'))
);
"""
adding_employees = """
INSERT INTO employee (name, surname, telephone, city,dep_id) VALUES 
('Max', 'Verstappen', '12762111', "Kielce", (SELECT dep_id FROM department WHERE dep_name = 'Electronics')),
('Lewis', 'Hamilton', '020056789', "Kielce", (SELECT dep_id FROM department WHERE dep_name = 'Electronics')),
('Oscar', 'Piastri', '123999888', "Kielce", (SELECT dep_id FROM department WHERE dep_name = 'Electronics'));
"""

commands = [changing_manager, deleting_employees, adding_employees]

for cmd in commands:
    execute_command(connection, cmd)

Command successful
Command successful
Command successful


Checking if the changes where effective:

In [29]:
print("Before:")
display(pd.read_sql("SELECT * FROM department WHERE dep_id = (SELECT dep_id FROM department WHERE dep_name = 'Electronics')", connection))
print("After:")
display(pd.read_sql("SELECT * FROM employee WHERE dep_id = (SELECT dep_id FROM department WHERE dep_name = 'Electronics')", connection))

Before:


,dep_id,dep_name,manager
0,1,Electronics,Michael Scott


After:


,soc_sec_num,name,surname,telephone,city,dep_id,salary_group
0,26,Max,Verstappen,12762111,Kielce,1,A
1,27,Lewis,Hamilton,020056789,Kielce,1,A
2,28,Oscar,Piastri,123999888,Kielce,1,A


***

## **4.Determine sales for a department in a particular period**

Very important feature of having a database is a ability to perform some data analysis on our sales. With this code , the following situation is simulated:

*The CEO what to determite what were the sales in each department in last weekend (11th-12th of March, 2026)*

In [30]:
sales_per_department = """

SELECT 
    d.dep_name AS Department, 
    COALESCE(SUM(p.price * b.quantity * c.discount),0) AS Total_sales

FROM department d

LEFT JOIN product p ON d.dep_id = p.dep_id
LEFT JOIN bill b ON p.product_num = b.product_num 
    AND b.date >= '2026-03-11' 
    AND b.date <= '2026-03-12'
LEFT JOIN customers c ON b.card_num = c.card_num


GROUP BY d.dep_name
ORDER BY total_sales DESC;

"""
pd.read_sql(sales_per_department, connection)

,Department,Total_sales
0,Electronics,1200.5
1,Clothing,194.0
2,Garden,180.0
3,Toys,89.0
4,Bakery,49.8
5,Cosmetics,33.5
6,Dairy,0.0


***

## **5.Determine the best-selling products in a particular department**

Next importnat business decision is to choose which product are woth investing in. The CEO wants to know which product is the most popular within each of departments.With this thing is a bit more complicated because I have to use subquery - First I create "table" with select stetment then I filter it with subquery, It would be impossible to do it in one query because we wiuld try to filter table by the column that does not yet exist. The following SQL query can gest us this information: 

In [31]:
best_selling_product = """

SELECT Department, Product, Quantity
FROM (
    SELECT 
        d.dep_name AS Department, 
        p.product_name AS Product, 
        SUM(b.quantity) AS Quantity,
        RANK() OVER (PARTITION BY d.dep_name ORDER BY SUM(b.quantity) DESC) as Ranking
    FROM department d 
    LEFT JOIN product p ON d.dep_id = p.dep_id
    LEFT JOIN bill b ON p.product_num = b.product_num
    GROUP BY d.dep_name, p.product_name
) 
WHERE Ranking = 1
ORDER BY Quantity DESC;
"""

pd.read_sql(best_selling_product, connection)

,Department,Product,Quantity
0,Garden,Fertilizer,10
1,Bakery,Baguette,5
2,Clothing,Leather Belt,3
3,Cosmetics,Lip Balm,3
4,Dairy,Organic Milk,3
5,Toys,Barbie Doll,2
6,Electronics,Smartphone,1
7,Electronics,Laptop,1
8,Electronics,Bluetooth Headphones,1


***

##  **6.Change the price of a product**

All of the information in database can be easily acessed and also updated. For example, the store wants to lower prices of imported products by 20% to make more customares buy them. The following code updates prices of products 

In [32]:
change_price = """
UPDATE product
SET price = ROUND(price * 0.8, 2)
WHERE origins = 'Imported';
"""
print("Before:")
display(pd.read_sql("SELECT product_name, ROUND(price, 2) as price, origins FROM product WHERE origins = 'Imported' LIMIT 3", con = connection )  )
execute_command(connection, change_price)
print("After:")
display(pd.read_sql("SELECT product_name, price, origins FROM product WHERE origins = 'Imported' LIMIT 3", con = connection ))

Before:


,product_name,price,origins
0,Greek Yogurt,1.5,Imported
1,Butter,4.0,Imported
2,Baguette,1.8,Imported


Command successful
After:


,product_name,price,origins
0,Greek Yogurt,1.20,Imported
1,Butter,3.20,Imported
2,Baguette,1.44,Imported


***

## **7.Change the number of products in stock;**

After supply of products its importnat to update data. The following code updates inventory of Bakery depertment after morning supply:

In [33]:
bakery_supply = """
UPDATE product
SET stock = stock + 350
WHERE dep_id = (SELECT dep_id FROM department WHERE dep_name = 'Bakery');
"""

print("Before:")
display(pd.read_sql("SELECT product_name, stock FROM product WHERE dep_id = (SELECT dep_id FROM department WHERE dep_name = 'Bakery') LIMIT 3", con = connection ))
execute_command(connection, bakery_supply)
print("After:")
display(pd.read_sql("SELECT product_name, stock FROM product WHERE dep_id = (SELECT dep_id FROM department WHERE dep_name = 'Bakery') LIMIT 3", con = connection ))

Before:


,product_name,stock
0,Rye Bread,50
1,Croissant,30
2,Baguette,40


Command successful
After:


,product_name,stock
0,Rye Bread,400
1,Croissant,380
2,Baguette,390


***

## **8.Show all the products which name starts with “A”**

For some unsure reason you might wanna see all of the product names started with "A", maybe the CEO looks for particular product but only remember that it started with "A". The task is not hard, the following code will filter all of the products with "A" as the firt letter. Unfortunetly in our database the products starting with A are not that popular

In [34]:
products_A = """
SELECT p.product_name AS Name
FROM product p
WHERE p.product_name LIKE 'A%'
"""

pd.read_sql(products_A, connection)

,Name
0,Apple Pie


***

## **9.Change an employee’s department and city**

Migriatiopn of employees is a standart thing happening all the time, it good to know how easily enter this information into the database. Recently Toby Flenderson from Cosmetics department moved to Bakery department (as he always wanted)and also changed his place of living to Kielce. The following code adresses this change;

In [35]:

chnaging_dep_city = """
UPDATE employee 
SET 
    city = 'Kielce', 
    dep_id = (SELECT dep_id FROM department WHERE dep_name = 'Bakery')
WHERE name = 'Toby' 
  AND surname = 'Flenderson' 
  AND dep_id = (SELECT dep_id FROM department WHERE dep_name = 'Cosmetics');
"""
print("Before:")
display(pd.read_sql("SELECT * FROM employee WHERE name = 'Toby' AND surname = 'Flenderson'", connection)    )
execute_command(connection, chnaging_dep_city)
print("After:")
display(pd.read_sql("SELECT * FROM employee WHERE name = 'Toby' AND surname = 'Flenderson'", connection)    )

Before:


,soc_sec_num,name,surname,telephone,city,dep_id,salary_group
0,21,Toby,Flenderson,333000111,Los Angeles,7,A


Command successful
After:


,soc_sec_num,name,surname,telephone,city,dep_id,salary_group
0,21,Toby,Flenderson,333000111,Kielce,3,A


***

## **10.Determine the most purchased products by a customer**

With database there is also an option for anylizing customares preferences like most liked products. The folowing code is loking fot the most purchesed product for each customare.

In [36]:
most_popular_products = """
SELECT Name, Surname, Product, Quantity
FROM
(SELECT 
    c.name AS Name, c.surname as Surname,
    p.product_name AS Product, SUM(b.quantity) AS Quantity,
    RANK() OVER (PARTITION by c.name ORDER BY SUM(b.quantity) DESC) as Ranking

FROM customers c 
LEFT JOIN bill b ON c.card_num = b.card_num
LEFT JOIN product p ON b.product_num = p.product_num

GROUP BY c.card_num, p.product_name ) 
WHERE Ranking = 1
ORDER BY Quantity DESC;
"""

pd.read_sql(most_popular_products, connection)

,Name,Surname,Product,Quantity
0,Felicia,Sweet,Fertilizer,10
1,Charles,Peace,Baguette,5
2,Edward,Scissorhands,Rye Bread,4
3,Beata,Knight,Organic Milk,3
4,Dorothy,Wellman,Leather Belt,3
5,Adam,Walker,Shovel,2
6,Gregory,Miller,Lip Balm,2


***

## **11.Determine the total expense made by a customer in a given period**

Nest with customare analysis we can anylize therirs expenses in given perioids. The following code will sumarize the expediture for each customare in the 11 - 12th march weekend


In [37]:
total_exp_customare = """
SELECT c.name AS Name, c.surname AS Surname,
ROUND(COALESCE(SUM(p.price * b.quantity * c.discount), 0), 2) AS Expenditure
FROM customers c
LEFT JOIN bill b ON c.card_num = b.card_num
    AND b.date >= '2026-03-11'
    AND b.date <= '2026-03-12'
LEFT JOIN product p ON b.product_num = p.product_num
GROUP BY c.card_num
ORDER BY Expenditure DESC;
"""

pd.read_sql(total_exp_customare, connection)

,Name,Surname,Expenditure
0,Dorothy,Wellman,1394.5
1,Felicia,Sweet,155.6
2,Edward,Scissorhands,112.0
3,Charles,Peace,31.2
4,Adam,Walker,0.0
5,Beata,Knight,0.0
6,Gregory,Miller,0.0


***

## **12. Determine the number of employees in a department.**

Database can also help with managing the shop, and things like employment. Lets see where we have what number of emploeyys.  

In [38]:
num_of_employees = """
SELECT dep_name AS Department, COUNT(soc_sec_num) AS Number_of_employees
FROM department d
LEFT JOIN employee e ON d.dep_id = e.dep_id
GROUP BY d.dep_name;"""

pd.read_sql(num_of_employees, connection)

,Department,Number_of_employees
0,Bakery,5
1,Clothing,5
2,Cosmetics,5
3,Dairy,2
4,Electronics,3
5,Garden,3
6,Toys,2


***

## **13. Further employment analysis**

Beside knowing how much people are employed in each department we can also calculate how much per year we are paying for them. Reminder: 

Salary groups:

**A level** - 24.000$/year, **B level** - 28.000$/year and **C level** - 32.000$/year

 The following code will sumarise the expednditure for employment and how much of each salary group is in each department:


In [39]:
employment_exp_per_department = """
SELECT d.dep_name AS Department, 
       COALESCE(SUM(CASE e.salary_group 
                    WHEN 'A' THEN 24000 
                    WHEN 'B' THEN 28000 
                    WHEN 'C' THEN 32000 
                    ELSE 0 END), 0) AS Total_Employment_Expense,
       COUNT(CASE WHEN e.salary_group = 'A' THEN 1 END) AS A_Group,
       COUNT(CASE WHEN e.salary_group = 'B' THEN 1 END) AS B_Group,
       COUNT(CASE WHEN e.salary_group = 'C' THEN 1 END) AS C_Group,
       COUNT(soc_sec_num) AS Total_Employees
FROM department d
LEFT JOIN employee e ON d.dep_id = e.dep_id
GROUP BY d.dep_name
ORDER BY Total_Employment_Expense DESC;
"""

pd.read_sql(employment_exp_per_department, connection)

,Department,Total_Employment_Expense,A_Group,B_Group,C_Group,Total_Employees
0,Clothing,136000,1,4,0,5
1,Cosmetics,128000,3,2,0,5
2,Bakery,124000,4,1,0,5
3,Garden,72000,3,0,0,3
4,Electronics,72000,3,0,0,3
5,Dairy,56000,1,0,1,2
6,Toys,48000,2,0,0,2


***

## **14. Further customare analysis**

Another instresting thing we can anlyze is if the customares have any preferences in the type of products that they are buying. The following code will summarize the quantity and expenditure for every client inside each origin group for products:

In [40]:
customare_by_origin = """
SELECT 
c.name AS Name, c.surname AS Surname, p.origins AS Product_Origin ,
COALESCE(SUM(b.quantity), 0) AS Quantity,
COALESCE(ROUND(SUM(p.price * b.quantity * c.discount), 2), 0) AS Expenditure

FROM customers c
CROSS JOIN (SELECT DISTINCT origins FROM product WHERE origins IS NOT NULL) AS origins_list
LEFT JOIN product p ON p.origins = origins_list.origins
LEFT JOIN bill b ON b.product_num = p.product_num AND b.card_num = c.card_num
GROUP BY c.card_num, origins_list.origins
ORDER BY c.card_num, origins_list.origins;
"""

pd.read_sql(customare_by_origin, connection)

,Name,Surname,Product_Origin,Quantity,Expenditure
0,Adam,Walker,Imported,0,0.00
1,Adam,Walker,Local,3,789.26
2,Beata,Knight,Imported,0,0.00
3,Beata,Knight,Local,5,17.29
4,Charles,Peace,Imported,5,7.20
5,Charles,Peace,Local,2,24.00
6,Dorothy,Wellman,Imported,0,0.00
7,Dorothy,Wellman,Local,5,1394.50
8,Edward,Scissorhands,Imported,2,51.20
9,Edward,Scissorhands,Local,6,60.80


***

## **15.Further Department analysis**

Now we want to know what type of products are most popular within the department.

In [41]:
department_by_origin = """
SELECT 
d.dep_name , p.origins AS Product_Origin ,
COALESCE(SUM(b.quantity), 0) AS Quantity,
COALESCE(ROUND(SUM(p.price * b.quantity * c.discount),2), 0) AS Expenditure

FROM department d
CROSS JOIN (SELECT DISTINCT origins FROM product WHERE origins IS NOT NULL) AS origins_list
LEFT JOIN product p ON p.origins = origins_list.origins AND p.dep_id = d.dep_id
LEFT JOIN bill b ON b.product_num = p.product_num 
LEFT JOIN customers c ON b.card_num = c.card_num
GROUP BY d.dep_name, origins_list.origins
ORDER BY d.dep_name, origins_list.origins;
"""

pd.read_sql(department_by_origin, connection)


,dep_name,Product_Origin,Quantity,Expenditure
0,Bakery,Imported,5,7.20
1,Bakery,Local,6,40.80
2,Clothing,Imported,0,0.00
3,Clothing,Local,5,205.88
4,Cosmetics,Imported,2,26.80
5,Cosmetics,Local,3,7.13
6,Dairy,Imported,0,0.00
7,Dairy,Local,4,14.91
8,Electronics,None,0,0.00
9,Electronics,Local,3,2045.04


In [42]:
connection.close()